In [8]:
WORK_DIR = "/Users/morgachev/workspace/gonka/governance-1"
APP_PATH = f"{WORK_DIR}/inferenced"
NODE_URL = "http://node2.gonka.ai:8000/chain-rpc/"

import subprocess
import json

def get_inference_param():
    cmd = f"{APP_PATH} query inference params --node {NODE_URL} -o json"
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)

def get_restriction_param():
    cmd = f"{APP_PATH} query restrictions params --node {NODE_URL} -o json"
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)

def get_mint_param():
    cmd = f"{APP_PATH} query mint params --node {NODE_URL} -o json"
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)

def wrap_in_message(
    data: dict,
    msg_type: str = "/inference.inference.MsgUpdateParams",
    authority: str = "gonka10d07y265gmmuvt4z0w9aw880jnsr700j2h5m33",
):
    msg = {
        "@type": msg_type,
        "authority": authority,
        "params": data["params"]
    }
    return msg


def wrap_inference_message(data: dict):
    return wrap_in_message(data, "/inference.inference.MsgUpdateParams")

def wrap_restriction_message(data: dict):
    return wrap_in_message(data, "/inference.restrictions.MsgUpdateParams")

def wrap_mint_message(data: dict):
    return wrap_in_message(data, "/cosmos.mint.v1beta1.MsgUpdateParams")

In [ ]:
inference_param = get_inference_param()
inference_param["params"]["epoch_params"]["epoch_length"] = str(15391)
inference_msg = wrap_inference_message(inference_param)

restriction_param = get_restriction_param()
restriction_param["params"]["restriction_end_block"] = str(1385263)
restriction_msg = wrap_restriction_message(restriction_param)

mint_param = get_mint_param()
mint_param["params"]["blocks_per_year"] = str(5618012)
mint_msg = wrap_mint_message(mint_param)



messages = [
    inference_msg,
    restriction_msg,
    mint_msg
]

with open(f"{WORK_DIR}/draft_proposal.json", "r") as f:
    draft_proposal = json.load(f)

draft_proposal["messages"] = messages

with open(f"{WORK_DIR}/proposal.json", "w") as f:
    json.dump(draft_proposal, f, indent=4)
